# Breast Cancer Dataset - Model Evaluation

### [Breast Cancer Dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Citation

[1] Wolberg, Street, and Mangasarian, https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install numpy pandas matplotlib scikit-learn

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
%matplotlib inline

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
IN_FEATURES = 30
IN_FEATURES

In [ ]:
H1 = 32
H1

In [ ]:
OUT_FEATURES = 2
OUT_FEATURES

In [ ]:
TEST_SIZE = 0.3
TEST_SIZE

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.001
LEARNING_RATE

In [ ]:
EPOCHS = 200
EPOCHS

In [ ]:
BATCH_SIZE = 32
BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Load Dataset

In [ ]:
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.int64)
X_MEAN = X.mean(axis=0)
X_STD = X.std(axis=0)
X = (X - X_MEAN) / X_STD
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train.shape, X_test.shape

## Create Dataset and DataLoaders

In [ ]:
class NumpyDataset(torch.utils.data.Dataset):
    """
    A memory-efficient dataset that stores data as NumPy arrays.
    """

    def __init__(self, X, y, device):
        """
        Initialize the dataset with NumPy arrays.

        Parameters:
            X (np.ndarray): Feature array.
            y (np.ndarray): Label array.
            device (torch.device): Device to move tensors to.

        Returns:
            None
        """
        self.X = X
        self.y = y
        self.device = device

    def __len__(self):
        """
        Return the number of samples in the dataset.

        Parameters:
            None

        Returns:
            int: Number of samples.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Return a single sample as tensors on the device.

        Parameters:
            idx (int): Index of the sample.

        Returns:
            tuple: Feature and label tensors.
        """
        X_tensor = torch.tensor(self.X[idx]).float().to(self.device)
        y_tensor = torch.tensor(self.y[idx]).long().to(self.device)
        return X_tensor, y_tensor

In [ ]:
train_loader = DataLoader(NumpyDataset(X_train, y_train, DEVICE),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(NumpyDataset(X_test, y_test, DEVICE),
                         batch_size=BATCH_SIZE, shuffle=False)
len(train_loader), len(test_loader)

## Create Model

In [ ]:
class Model(nn.Module):
    """
    A small feedforward classifier for tabular data.
    """

    def __init__(self, in_features=IN_FEATURES, h1=H1, out_features=OUT_FEATURES):
        """
        Initialize the neural network layers.

        Parameters:
            in_features (int): Number of input features.
            h1 (int): Neurons in the hidden layer.
            out_features (int): Number of output features.

        Returns:
            None
        """
        super(Model, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, h1),
            nn.ReLU(),
            nn.Linear(h1, out_features))

    def forward(self, x):
        """
        Run the forward pass of the network.

        Parameters:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output logits.
        """
        return self.net(x)

## Instantiate Model, Loss, and Optimizer

In [ ]:
torch.manual_seed(SEED)
model = Model().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Train Model

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """
    Train the model for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        loss_fn (nn.Module): Loss function.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss.
    """
    model.train()
    total = 0.0
    for x, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)

In [ ]:
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d} | train loss {train_loss:.4f}")

## Collect Predictions and Probabilities

In [ ]:
model.eval()
all_probs = []
all_labels = []
with torch.no_grad():
    for x, y in test_loader:
        probs = torch.softmax(model(x), dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.cpu().numpy())
y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)
y_pred = (y_prob >= 0.5).astype(int)
print('collected', y_true.shape)

## Classification Metrics

### Functions

In [ ]:
def metric_table(y_true, y_pred):
    """
    Build a table of core classification metrics.

    Parameters:
        y_true (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.

    Returns:
        pandas.DataFrame: A single-row metrics table.
    """
    row = {'accuracy': accuracy_score(y_true, y_pred),
           'precision': precision_score(y_true, y_pred),
           'recall': recall_score(y_true, y_pred),
           'f1': f1_score(y_true, y_pred),
           'roc_auc': roc_auc_score(y_true, y_pred)}
    return pd.DataFrame(row, index=[0])

### Run

In [ ]:
metric_table(y_true, y_pred)

In [ ]:
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=list(data.target_names)))

## ROC Curve and AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_score = roc_auc_score(y_true, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend()
plt.tight_layout()
plt.show()

## Precision-Recall Curve

The precision-recall curve is more informative than ROC when the classes are imbalanced.

In [ ]:
precision, recall, _ = precision_recall_curve(y_true, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-recall curve')
plt.tight_layout()
plt.show()

## Calibration

A calibrated model that reports 0.8 confidence is correct about 80 percent of the time. The diagonal is perfect calibration.

In [ ]:
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)
plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker='o', label='Model')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect')
plt.xlabel('Predicted probability')
plt.ylabel('Observed frequency')
plt.title('Calibration curve')
plt.legend()
plt.tight_layout()
plt.show()

## Threshold Tuning

The default 0.5 threshold is not always best. Sweep the threshold and choose the F1-optimal cutoff.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)
scores = [f1_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
best_index = int(np.argmax(scores))
print(f"Best threshold: {thresholds[best_index]:.2f}")
print(f"Best F1: {scores[best_index]:.4f}")
plt.figure(figsize=(8, 4))
plt.plot(thresholds, scores, marker='o')
plt.xlabel('Threshold')
plt.ylabel('F1 score')
plt.title('F1 by decision threshold')
plt.tight_layout()
plt.show()

## Save Model

In [ ]:
torch.save(model.state_dict(), 'eval_breast_cancer.pt')
print('saved eval_breast_cancer.pt')

## Load Model

In [ ]:
loaded_model = Model().to(DEVICE)
loaded_model.load_state_dict(torch.load('eval_breast_cancer.pt', map_location=DEVICE))
loaded_model.eval()
print('loaded eval_breast_cancer.pt')

## Inference

### Function

In [ ]:
def predict(model, features, threshold=0.5):
    """
    Predict benign or malignant with a tunable threshold.

    Parameters:
        model (nn.Module): Trained model.
        features (list): Raw feature values.
        threshold (float): Decision threshold.

    Returns:
        tuple: Predicted label name and probability.
    """
    model.eval()
    scaled = (np.array(features, dtype=np.float32) - X_MEAN) / X_STD
    with torch.no_grad():
        X_new = torch.tensor(scaled).float().to(DEVICE).unsqueeze(0)
        probability = torch.softmax(model(X_new), dim=1)[0, 1].item()
    label = 'benign' if probability >= threshold else 'malignant'
    return label, probability

### Run Inference

In [ ]:
label, probability = predict(loaded_model, X_test[0])
print(f"Predicted: {label} (p={probability:.4f}) | Actual: {data.target_names[y_test[0]]}")